In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../data/rugby.db')
df = pd.read_sql_query("SELECT * FROM team_match_stats", conn)
matches = pd.read_sql_query("SELECT * FROM matches", conn)
teams = pd.read_sql_query("SELECT * FROM teams", conn)
conn.close()

print(df.shape)
df.head()

(472, 201)


,uid,matchEspnId,teamEspnId,opponentEspnId,linescore1stHalf,linescore2ndHalf,linescore20min,linescore60min,passes,runs,...,penaltyConcededWrongSide,won,lost,drawn,numberOfTeams,matches,startingMatches,replacementMatches,onReport,playTheBall
0,196b190c03dbd56b,597739,25912,25922,10,26,NaN,NaN,89.0,77.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,a4fbb6484a3891d3,597739,25922,25912,7,7,NaN,NaN,117.0,111.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,b36bef4da55690b2,597740,99855,143737,10,23,NaN,NaN,170.0,130.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,b43034f2cb29eb6d,597740,143737,99855,8,18,NaN,NaN,83.0,80.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,ebb612e8543e007b,597741,25920,25921,7,7,NaN,NaN,93.0,81.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [6]:
fill_rate = df.notna().mean().sort_values(ascending=False) * 100
print(fill_rate[fill_rate > 0].to_string())

uid                                 100.000000
teamEspnId                          100.000000
opponentEspnId                      100.000000
linescore1stHalf                    100.000000
linescore2ndHalf                    100.000000
matchEspnId                         100.000000
freeKickConcededInRuckOrMaul         79.237288
lineoutsInfringeOwn                  79.237288
lineoutsInfringeOpp                  79.237288
totalKicksSucceeded                  79.237288
totalFreeKicksConceded               79.237288
freeKickConcededAtLineout            79.237288
freeKickConcededInGeneralPlay        79.237288
freeKickConcededAtScrum              79.237288
lineoutsToOwnPlayer                  79.237288
freeKickConceded                     79.237288
lineoutsToOppPlayer                  79.237288
lineoutThrowLostFreeKick             79.237288
lineoutWonOwnThrow                   79.237288
lineoutWonSteal                      79.237288
tryKicks                             79.237288
lineoutThrowL

In [8]:
# Métriques de pression offensive
pression_cols = [
    'attackingEventsZoneC',  # événements dans les 22m adverses
    'attackingEventsZoneD',  # événements dans les 5m adverses
    'cleanBreaks',           # franchissements
    'defendersBeaten',       # défenseurs battus
    'metres',                # mètres gagnés
    'possession',            # possession
    'territory',             # territoire
    'carriesCrossedGainLine' # passages de la ligne d'avantage
]

# Normalisation 0-1 par colonne
from sklearn.preprocessing import MinMaxScaler
import numpy as np

df_clean = df[pression_cols].dropna()
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_clean), columns=pression_cols)

# Score composite
df_clean = df_clean.copy()
df_clean['OPI'] = df_scaled.mean(axis=1)

print(df_clean['OPI'].describe())
df_clean.head()

count    374.000000
mean       0.409217
std        0.128818
min        0.082163
25%        0.317875
50%        0.398167
75%        0.487821
max        0.784112
Name: OPI, dtype: float64


,attackingEventsZoneC,attackingEventsZoneD,cleanBreaks,defendersBeaten,metres,possession,territory,carriesCrossedGainLine,OPI
0,115.0,41.0,4.0,21.0,374.0,0.47,0.53,56.0,0.343383
1,107.0,84.0,4.0,24.0,382.0,0.53,0.47,61.0,0.382769
2,225.0,78.0,8.0,16.0,542.0,0.58,0.61,86.0,0.566811
3,82.0,47.0,3.0,10.0,341.0,0.42,0.39,49.0,0.220761
4,98.0,19.0,4.0,21.0,281.0,0.46,0.44,43.0,0.260066


In [9]:
# Trouver le match Toulouse vs Bordeaux-Bèglès
mask = matches['name'].str.contains('Toulousain') & matches['name'].str.contains('Bordeaux')
print(matches[mask][['espnId', 'name', 'date', 'winnerScore', 'loserScore']])

     espnId                                 name                 date  \
27   597766  Stade Toulousain vs Bordeaux Begles  2023-10-29 20:05:00   
127  597866  Bordeaux Begles vs Stade Toulousain  2024-03-24 20:05:00   
186  597925  Stade Toulousain vs Bordeaux Begles  2024-06-28 19:05:00   
213  599632  Stade Toulousain vs Bordeaux Begles  2024-09-29 19:05:00   

     winnerScore  loserScore  
27            29          22  
127           31          28  
186           59           3  
213           16          12  


je suid

## 9